In [8]:
import json
import re
from pathlib import Path
from copy import deepcopy

INPUT_PATH = "/home/vcnt/Repositories/Tesis/datasets_metadata/histai_breast_metadata.json"
OUTPUT_PATH = "/home/vcnt/Repositories/Tesis/datasets_metadata/histai_breast_metadata_regroup.json"

def normalize_text(s):
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = s.replace("mammae sinistrae", "left breast")
    s = s.replace("mammae dextrae", "right breast")
    s = s.replace("ca ", "cancer ")
    s = s.replace(" c-r ", " cancer ")
    s = s.replace(" cr ", " cancer ")
    s = s.replace("ibc", "invasive breast cancer")
    s = s.replace("idc", "invasive ductal carcinoma")
    s = s.replace("dcis", "ductal carcinoma in situ")
    s = re.sub(r'c50(\.\d+)?', 'breast cancer', s)
    s = re.sub(r'd24(\.\d+)?', 'benign breast neoplasm', s)
    s = re.sub(r'd48\.6', 'indeterminate breast neoplasm', s)
    s = re.sub(r'[^a-z0-9\s\?\-]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def detect_laterality(s):
    left = bool(re.search(r'\bleft\b|\bsinistrae\b', s))
    right = bool(re.search(r'\bright\b|\bdextrae\b', s))
    bilateral = bool(re.search(r'\bbilateral\b|right and left|left and right|both breasts', s))
    if bilateral:
        return "bilateral"
    if left and not right:
        return "left"
    if right and not left:
        return "right"
    if left and right:
        return "bilateral"
    return "unspecified"

def has_uncertainty(s):
    terms = [
        "?", "suspicious", "possible", "query", "suspected",
        "indeterminate", "cannot exclude", "vs", "differential diagnosis"
    ]
    return any(t in s for t in terms)

def classify_diagnosis(raw_dx):
    s = normalize_text(raw_dx)
    laterality = detect_laterality(s)

    if s in {"", "-", "breast", "breast unspecified part", "unspecified breast disease"}:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "missing_or_uninformative",
            "taxonomy_subgroup": "missing_or_blank",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "empty_or_uninformative"
        }

    if any(x in s for x in ["lung cancer", "nsclc", "ovarian neoplasm", "nose deformity"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "non_breast_or_metadata_noise",
            "taxonomy_subgroup": "non_breast_entity",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "non_breast_entity"
        }

    if any(x in s for x in ["sectoral resection", "mastectomy", "endoprosthesis", "implant", "replacement of tissue implants", "morphological verification"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "non_breast_or_metadata_noise",
            "taxonomy_subgroup": "procedure_or_anatomic_note",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "procedure_note"
        }

    if any(x in s for x in ["mastitis", "inflammatory diseases of the breast", "inflammatory"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "inflammatory_or_reactive",
            "taxonomy_subgroup": "mastitis_or_inflammatory",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "inflammatory"
        }

    if any(x in s for x in ["gynecomastia"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign",
            "taxonomy_subgroup": "gynecomastia",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "gynecomastia"
        }

    if any(x in s for x in ["fibroadenoma", "fibroadenomatosis"]):
        if has_uncertainty(s):
            return {
                "diagnosis_normalized": s,
                "taxonomy_group": "suspicious_or_uncertain",
                "taxonomy_subgroup": "query_benign_vs_malignant",
                "taxonomy_laterality": laterality,
                "taxonomy_status": "review",
                "taxonomy_rule": "fibroadenoma_uncertain"
            }
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign",
            "taxonomy_subgroup": "fibroadenoma",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "fibroadenoma"
        }

    if any(x in s for x in ["intraductal papilloma", "intraductal papillomas", "papillomatosis", "cystadenopapilloma", "cystoadenopapilloma", "mintz"]):
        if "papillary carcinoma" not in s:
            return {
                "diagnosis_normalized": s,
                "taxonomy_group": "benign",
                "taxonomy_subgroup": "intraductal_papilloma",
                "taxonomy_laterality": laterality,
                "taxonomy_status": "final",
                "taxonomy_rule": "papilloma"
            }

    if any(x in s for x in ["fibrocystic", "mastopathy", "fibrosclerosis", "dysplasia"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign",
            "taxonomy_subgroup": "fibrocystic_change_or_mastopathy",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "fibrocystic_mastopathy"
        }

    if any(x in s for x in ["adenosis"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign",
            "taxonomy_subgroup": "adenosis",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "adenosis"
        }

    if any(x in s for x in ["phyllodes sarcoma", "malignant mixed tumor", "malignant phyllodes"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "malignant_phyllodes_or_mixed",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "malignant_phyllodes"
        }

    if "phyllodes tumor" in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "atypical_or_borderline",
            "taxonomy_subgroup": "phyllodes_tumor_borderline_unspecified",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "review",
            "taxonomy_rule": "phyllodes_unspecified"
        }

    if any(x in s for x in ["paget"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "paget_disease",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "paget"
        }

    if any(x in s for x in ["ductal carcinoma in situ", "breast in situ carcinoma", "carcinoma in situ"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "ductal_carcinoma_in_situ",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "dcis"
        }

    if any(x in s for x in ["invasive ductal carcinoma", "infiltrating ductal carcinoma"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "invasive_ductal_carcinoma",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "idc"
        }

    if "lobular carcinoma" in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "lobular_carcinoma",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "lobular"
        }

    if "mucinous carcinoma" in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "mucinous_carcinoma",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "mucinous"
        }

    if "papillary carcinoma" in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "papillary_carcinoma",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "papillary_carcinoma"
        }

    if any(x in s for x in [
        "metastasis of breast cancer", "breast cancer metastasis",
        "metastatic breast cancer", "metastasis of invasive breast cancer",
        "metastatic to liver", "metastases to the lungs",
        "metastases to axillary lymph nodes", "secondary bone involvement"
    ]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "metastatic_breast_cancer",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "metastatic"
        }

    if any(x in s for x in [
        "invasive breast cancer", "invasive breast carcinoma",
        "invasive mammary carcinoma", "invasive carcinoma",
        "infiltrating carcinoma", "invasive unspecified breast cancer",
        "invasive unspecified breast carcinoma", "breast invasive carcinoma",
        "breast invasive malignancy", "invasive malignant neoplasm of the breast"
    ]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "invasive_carcinoma_nos",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "invasive_carcinoma_nos"
        }

    if any(x in s for x in [
        "breast cancer", "breast carcinoma", "carcinoma of the breast",
        "malignant neoplasm of the breast", "breast malignancy",
        "malignant breast neoplasm", "malignant mammary tumor",
        "cancer of the left breast", "cancer of the right breast",
        "left breast cancer", "right breast cancer"
    ]):
        if has_uncertainty(s):
            return {
                "diagnosis_normalized": s,
                "taxonomy_group": "suspicious_or_uncertain",
                "taxonomy_subgroup": "suspicious_for_malignancy",
                "taxonomy_laterality": laterality,
                "taxonomy_status": "review",
                "taxonomy_rule": "cancer_uncertain"
            }
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "breast_carcinoma_nos",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "final",
            "taxonomy_rule": "breast_cancer_nos"
        }

    if any(x in s for x in [
        "indeterminate breast neoplasm", "indeterminate neoplasm",
        "suspicious malignant neoplasm"
    ]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "atypical_or_borderline",
            "taxonomy_subgroup": "indeterminate_neoplasm",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "review",
            "taxonomy_rule": "indeterminate_neoplasm"
        }

    if has_uncertainty(s):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "suspicious_or_uncertain",
            "taxonomy_subgroup": "suspicious_for_malignancy",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "review",
            "taxonomy_rule": "generic_uncertainty"
        }

    if any(x in s for x in [
        "lesion", "mass", "tumor", "formation", "neoplasm", "nodule", "microcalcifications"
    ]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "nonspecific_lesion_or_mass",
            "taxonomy_subgroup": "breast_mass_or_lesion_nos",
            "taxonomy_laterality": laterality,
            "taxonomy_status": "review",
            "taxonomy_rule": "nonspecific_mass_lesion"
        }

    return {
        "diagnosis_normalized": s,
        "taxonomy_group": "non_breast_or_metadata_noise",
        "taxonomy_subgroup": "unmapped_review",
        "taxonomy_laterality": laterality,
        "taxonomy_status": "review",
        "taxonomy_rule": "fallback_unmapped"
    }

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

out = []
for row in data:
    row2 = deepcopy(row)
    mapped = classify_diagnosis(row.get("diagnosis", ""))
    row2.update(mapped)
    out.append(row2)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f"Saved {len(out)} records to {OUTPUT_PATH}")

Saved 1492 records to /home/vcnt/Repositories/Tesis/datasets_metadata/histai_breast_metadata_regroup.json
